In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen3-1.7B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': '<think>\nOkay, the user is asking "Who are you?" I need to respond in a friendly and engaging way. Let me start by introducing myself as an AI assistant. I should mention that I\'m designed to help with various tasks and provide information. It\'s important to highlight that I\'m here to assist and that I\'m continuously learning. I should also invite them to ask questions so they feel comfortable sharing their needs. Need to keep the tone positive and approachable. Let me put that all together clearly and concisely.\n</think>\n\nHello! I\'m an AI assistant designed to help you with your questions and tasks. I can provide information, answer questions, and assist with various tasks. I\'m here to make your day easier and more helpful. Feel free to ask me anything you need! 😊'}]}]

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import json, re


tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B")


def generate_bio(prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt + " /no_think"}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    return re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()


# English
en_bio = generate_bio("Write a biography of Sheikh Hasina.")
print("ENGLISH:\n", en_bio)


# Bengali
bn_bio = generate_bio("শেখ হাসিনার একটি জীবনী বাংলায় লিখুন।")
print("\nBENGALI:\n", bn_bio)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ENGLISH:
 Sheikh Hasina Mohsin, often referred to as Sheikh Hasina, is a prominent political leader from Bangladesh, serving as the Prime Minister of Bangladesh since 2009. She is the first woman to hold the position of Prime Minister in the country's history and one of the most influential political figures in South Asia.

### Early Life

She was born on 12 April 1961 in the town of Chittagong, Bangladesh. She grew up in a family of modest means, and her early education was provided by the local government. She studied at the University of Dhaka, where she earned a degree in Economics. She later pursued a master's degree in Political Science from the University of London.

### Political Career

Hasina began her political career in the 1980s, joining the Bangladesh Nationalist Party (BNP), a major political party in Bangladesh. She was elected to the National Assembly in 1986 and later became a member of the BNP's leadership. In 1990, she was elected as the first woman to serve as the 

In [ ]:
import os
os.environ['HF_TOKEN'] = 'YOUR_TOKEN_HERE'

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

completion = client.chat.completions.create(
    model="Qwen/Qwen3-1.7B",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)